### N-gram language models or how to write scientific papers (4 pts)

We shall train our language model on a corpora of [ArXiv](http://arxiv.org/) articles and see if we can generate a new one!

![img](https://media.npr.org/assets/img/2013/12/10/istock-18586699-monkey-computer_brick-16e5064d3378a14e0e4c2da08857efe03c04695e-s800-c85.jpg)

_data by neelshah18 from [here](https://www.kaggle.com/neelshah18/arxivdataset/)_

_Disclaimer: this has nothing to do with actual science. But it's fun, so who cares?!_

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
import urllib.request

url = "https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1"
urllib.request.urlretrieve(url, "./arxivData.json.tar.gz")
!tar -xvzf arxivData.json.tar.gz
data = pd.read_json("./arxivData.json")
data.sample(n=5)

x arxivData.json


,author,day,id,link,month,summary,tag,title,year
8439,"[{'name': 'Peng Tang'}, {'name': 'Xinggang Wan...",31,1608.00182v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",7,Despite the great success of convolutional neu...,"[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",Deep FisherNet for Object Classification,2016
19322,[{'name': 'Denis Berthier'}],11,1304.3208v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",4,Many real world problems naturally appear as c...,"[{'term': 'cs.AI', 'scheme': 'http://arxiv.org...","From Constraints to Resolution Rules, Part I: ...",2013
1332,"[{'name': 'Scott Reed'}, {'name': 'Honglak Lee...",20,1412.6596v3,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",12,Current state-of-the-art deep learning systems...,"[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",Training Deep Neural Networks on Noisy Labels ...,2014
3696,"[{'name': 'Amit Daniely'}, {'name': 'Roy Frost...",18,1602.05897v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",2,We develop a general duality between neural ne...,"[{'term': 'cs.LG', 'scheme': 'http://arxiv.org...",Toward Deeper Understanding of Neural Networks...,2016
36301,"[{'name': 'Venkat Chandrasekaran'}, {'name': '...",13,1206.3240v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",6,It is well-known that inference in graphical m...,"[{'term': 'cs.DS', 'scheme': 'http://arxiv.org...",Complexity of Inference in Graphical Models,2012


In [3]:
# assemble lines: concatenate title and description
lines = data.apply(lambda row: row['title'] + ' ; ' + row['summary'].replace("\n", ' '), axis=1).tolist()

sorted(lines, key=len)[:3]

['Differential Contrastive Divergence ; This paper has been retracted.',
 'What Does Artificial Life Tell Us About Death? ; Short philosophical essay',
 'P=NP ; We claim to resolve the P=?NP problem via a formal argument for P=NP.']

### Tokenization

You know the dril. The data is messy. Go clean the data. Use WordPunctTokenizer or something.


In [4]:
# Task: convert lines (in-place) into strings of space-separated tokens. Import & use WordPunctTokenizer
from nltk.tokenize import WordPunctTokenizer

tokenizer = WordPunctTokenizer()
lines = [" ".join(tokenizer.tokenize(line.lower())) for line in lines]

In [5]:
assert sorted(lines, key=len)[0] == \
    'differential contrastive divergence ; this paper has been retracted .'
assert sorted(lines, key=len)[2] == \
    'p = np ; we claim to resolve the p =? np problem via a formal argument for p = np .'

### N-Gram Language Model (1point)

A language model is a probabilistic model that estimates text probability: the joint probability of all tokens $w_t$ in text $X$: $P(X) = P(w_1, \dots, w_T)$.

It can do so by following the chain rule:
$$P(w_1, \dots, w_T) = P(w_1)P(w_2 \mid w_1)\dots P(w_T \mid w_1, \dots, w_{T-1}).$$ 

The problem with such approach is that the final term $P(w_T \mid w_1, \dots, w_{T-1})$ depends on $n-1$ previous words. This probability is impractical to estimate for long texts, e.g. $T = 1000$.

One popular approximation is to assume that next word only depends on a finite amount of previous words:

$$P(w_t \mid w_1, \dots, w_{t - 1}) = P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1})$$

Such model is called __n-gram language model__ where n is a parameter. For example, in 3-gram language model, each word only depends on 2 previous words. 

$$
    P(w_1, \dots, w_n) = \prod_t P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1}).
$$

You can also sometimes see such approximation under the name of _n-th order markov assumption_.

The first stage to building such a model is counting all word occurences given N-1 previous words

In [6]:
from tqdm import tqdm
from collections import defaultdict, Counter

# special tokens: 
# - `UNK` represents absent tokens, 
# - `EOS` is a special token after the end of sequence

UNK, EOS = "_UNK_", "_EOS_"

def count_ngrams(lines, n):
    """
    Count how many times each word occured after (n - 1) previous words
    :param lines: an iterable of strings with space-separated tokens
    :returns: a dictionary { tuple(prefix_tokens): {next_token_1: count_1, next_token_2: count_2}}

    When building counts, please consider the following two edge cases:
    - if prefix is shorter than (n - 1) tokens, it should be padded with UNK. For n=3,
      empty prefix: "" -> (UNK, UNK)
      short prefix: "the" -> (UNK, the)
      long prefix: "the new approach" -> (new, approach)
    - you should add a special token, EOS, at the end of each sequence
      "... with deep neural networks ." -> (..., with, deep, neural, networks, ., EOS)
      count the probability of this token just like all others.
    """
    counts = defaultdict(Counter)
    shift = n - 1
    lines = [(UNK + " ") * shift + line + " " + EOS for line in lines]

    for line in tqdm(lines):
        line_list = line.split()
        for i, token in enumerate(line_list[shift:]):
            prefix = tuple(line_list[i: i + shift])
            counts[prefix][token] += 1

    return counts

In [7]:
# let's test it
dummy_lines = sorted(lines, key=len)[:100]
dummy_counts = count_ngrams(dummy_lines, n=3)
assert set(map(len, dummy_counts.keys())) == {2}, "please only count {n-1}-grams"
assert len(dummy_counts[('_UNK_', '_UNK_')]) == 78
assert dummy_counts['_UNK_', 'a']['note'] == 3
assert dummy_counts['p', '=']['np'] == 2
assert dummy_counts['author', '.']['_EOS_'] == 1

100%|██████████| 100/100 [00:00<00:00, 14979.66it/s]


Once we can count N-grams, we can build a probabilistic language model.
The simplest way to compute probabilities is in proporiton to counts:

$$ P(w_t | prefix) = { Count(prefix, w_t) \over \sum_{\hat w} Count(prefix, \hat w) } $$

In [16]:
class NGramLanguageModel:    
    def __init__(self, lines, n):
        """ 
        Train a simple count-based language model: 
        compute probabilities P(w_t | prefix) given ngram counts
        
        :param n: computes probability of next token given (n - 1) previous words
        :param lines: an iterable of strings with space-separated tokens
        """
        assert n >= 1
        self.n = n
        self.shift = n - 1
    
        counts = count_ngrams(lines, self.n)
        
        # compute token probabilities given counts
        self.probs = defaultdict(Counter)
        for prefix, token_counter in counts.items():
            prefix_counts = token_counter.total()
            for token, token_count in token_counter.items():
                self.probs[prefix][token] = token_count / prefix_counts

    def _get_prefix(self, prefix):
        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - self.shift):]
        prefix = [UNK] * (self.shift - len(prefix)) + prefix

        return tuple(prefix)
            
    def get_possible_next_tokens(self, prefix):
        """
        :param prefix: string with space-separated prefix tokens
        :returns: a dictionary {token : it's probability} for all tokens with positive probabilities
        """
        return self.probs[self._get_prefix(prefix)]
    
    def get_next_token_prob(self, prefix, next_token):
        """
        :param prefix: string with space-separated prefix tokens
        :param next_token: the next token to predict probability for
        :returns: P(next_token|prefix) a single number, 0 <= P <= 1
        """
        return self.get_possible_next_tokens(prefix).get(next_token, 0)

Let's test it!

In [9]:
dummy_lm = NGramLanguageModel(dummy_lines, n=3)

p_initial = dummy_lm.get_possible_next_tokens('') # '' -> ['_UNK_', '_UNK_']
assert np.allclose(p_initial['learning'], 0.02)
assert np.allclose(p_initial['a'], 0.13)
assert np.allclose(p_initial.get('meow', 0), 0)
assert np.allclose(sum(p_initial.values()), 1)

p_a = dummy_lm.get_possible_next_tokens('a') # '' -> ['_UNK_', 'a']
assert np.allclose(p_a['machine'], 0.15384615)
assert np.allclose(p_a['note'], 0.23076923)
assert np.allclose(p_a.get('the', 0), 0)
assert np.allclose(sum(p_a.values()), 1)

assert np.allclose(dummy_lm.get_possible_next_tokens('a note')['on'], 1)
assert dummy_lm.get_possible_next_tokens('a machine') == \
    dummy_lm.get_possible_next_tokens("there have always been ghosts in a machine"), \
    "your 3-gram model should only depend on 2 previous words"

100%|██████████| 100/100 [00:00<00:00, 16196.10it/s]


Now that you've got a working n-gram language model, let's see what sequences it can generate. But first, let's train it on the whole dataset.

In [10]:
lm = NGramLanguageModel(lines, n=3)

100%|██████████| 41000/41000 [00:13<00:00, 3104.92it/s]


The process of generating sequences is... well, it's sequential. You maintain a list of tokens and iteratively add next token by sampling with probabilities.

$ X = [] $

__forever:__
* $w_{next} \sim P(w_{next} | X)$
* $X = concat(X, w_{next})$


Instead of sampling with probabilities, one can also try always taking most likely token, sampling among top-K most likely tokens or sampling with temperature. In the latter case (temperature), one samples from

$$w_{next} \sim {P(w_{next} | X) ^ {1 / \tau} \over \sum_{\hat w} P(\hat w | X) ^ {1 / \tau}}$$

Where $\tau > 0$ is model temperature. If $\tau << 1$, more likely tokens will be sampled with even higher probability while less likely tokens will vanish.

In [11]:
lm.get_possible_next_tokens('there have')

Counter({'been': 0.9083969465648855,
         'not': 0.03816793893129771,
         'only': 0.015267175572519083,
         'also': 0.015267175572519083,
         'lately': 0.007633587786259542,
         'very': 0.007633587786259542,
         'occurred': 0.007633587786259542})

In [8]:
import random

def get_next_token(lm, prefix, temperature=1.0):
    """
    return next token after prefix;
    :param temperature: samples proportionally to lm probabilities ^ (1 / temperature)
        if temperature == 0, always takes most likely token. Break ties arbitrarily.
    """
    pos_next_tokens = lm.get_possible_next_tokens(prefix)
    if temperature == 0:
        return sorted(pos_next_tokens, key=pos_next_tokens.get, reverse=True)[0]

    power = 1.0 / temperature
    sum_probs = sum([proba ** power for proba in pos_next_tokens.values()])
    tokens_probs = {token: proba ** power / sum_probs for token, proba in pos_next_tokens.items()}

    tokens = list(tokens_probs.keys())
    weights = list(tokens_probs.values())
    return random.choices(tokens, weights=weights, k=1)[0]

In [ ]:
from collections import Counter
test_freqs = Counter([get_next_token(lm, 'there have') for _ in range(10000)])
assert 250 < test_freqs['not'] < 450
assert 8500 < test_freqs['been'] < 9500
assert 1 < test_freqs['lately'] < 200

test_freqs = Counter([get_next_token(lm, 'deep', temperature=1.0) for _ in range(10000)])
assert 1500 < test_freqs['learning'] < 3000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.5) for _ in range(10000)])
assert 8000 < test_freqs['learning'] < 9000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.0) for _ in range(10000)])
assert test_freqs['learning'] == 10000

print("Looks nice!")

Let's have fun with this model

In [14]:
prefix = 'artificial' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break
        
print(prefix)

artificial intelligence ( ai ) algorithm , this is minimised . we explain how complex network topologies revealed that the lowest error rate ( wer ) improvements of cnn and fusion for splicing site , or more 2d training data that is multi - layer random neural network architecture , we propose a learning step to form flexible nonlinear tensor decomposition ; this paper , we propose a novel dataset called ug ^ 2 $ penalty . in this paper presents the results of the israeli - palestinian conflict is based on bregman iterations ; it couples with the era of


In [15]:
prefix = 'bridging the' # <- more of your ideas

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.5)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break
        
print(prefix)

bridging the gap between the training data . we also show that the proposed method can effectively learn the subspace of the art on several datasets . _EOS_


__More in the homework:__ nucleus sampling, top-k sampling, beam search(not for the faint of heart).

### Evaluating language models: perplexity (1point)

Perplexity is a measure of how well your model approximates the true probability distribution behind the data. __Smaller perplexity = better model__.

To compute perplexity on one sentence, use:
$$
    {\mathbb{P}}(w_1 \dots w_N) = P(w_1, \dots, w_N)^{-\frac1N} = \left( \prod_t P(w_t \mid w_{t - n}, \dots, w_{t - 1})\right)^{-\frac1N},
$$


On the corpora level, perplexity is a product of probabilities of all tokens in all sentences to the power of $1/N$, where $N$ is __total length (in tokens) of all sentences__ in corpora.

This number can quickly get too small for float32/float64 precision, so we recommend you to first compute log-perplexity (from log-probabilities) and then take the exponent.

In [9]:
def perplexity(lm, lines, min_logprob=np.log(10 ** -50.)):
    """
    :param lines: a list of strings with space-separated tokens
    :param min_logprob: if log(P(w | ...)) is smaller than min_logprop, set it equal to min_logrob
    :returns: corpora-level perplexity - a single scalar number from the formula above
    
    Note: do not forget to compute P(w_first | empty) and P(eos | full_sequence)
    
    PLEASE USE lm.get_next_token_prob and NOT lm.get_possible_next_tokens
    """
    logprobs = []
    shift = lm.shift
    lines = [(UNK + " ") * shift + line + " " + EOS for line in lines]

    for line in lines:
        line_list = line.split()
        for i, token in enumerate(line_list[shift:]):
            prefix = " ".join(line_list[i: i + shift])
            proba = np.log(lm.get_next_token_prob(prefix, token))
            proba = max(proba, min_logprob)
            logprobs.append(proba)

    ppx = -np.mean(logprobs)
    ppx = np.exp(ppx)
    
    return ppx

In [17]:
lm1 = NGramLanguageModel(dummy_lines, n=1)
lm3 = NGramLanguageModel(dummy_lines, n=3)
lm10 = NGramLanguageModel(dummy_lines, n=10)

ppx1 = perplexity(lm1, dummy_lines)
ppx3 = perplexity(lm3, dummy_lines)
ppx10 = perplexity(lm10, dummy_lines)
ppx_missing = perplexity(lm3, ['the jabberwock , with eyes of flame , '])  # thanks, L. Carrol

print("Perplexities: ppx1=%.3f ppx3=%.3f ppx10=%.3f" % (ppx1, ppx3, ppx10))

assert all(0 < ppx < 500 for ppx in (ppx1, ppx3, ppx10)), "perplexity should be non-negative and reasonably small"
assert ppx1 > ppx3 > ppx10, "higher N models should overfit and "
assert np.isfinite(ppx_missing) and ppx_missing > 10 ** 6, "missing words should have large but finite perplexity. " \
    " Make sure you use min_logprob right"
assert np.allclose([ppx1, ppx3, ppx10], (318.2132342216302, 1.5199996213739575, 1.1838145037901249))

100%|██████████| 100/100 [00:00<00:00, 18727.08it/s]

Perplexities: ppx1=318.213 ppx3=1.520 ppx10=1.184



C:\Users\lymuthien\AppData\Local\Temp\ipykernel_14216\4221468691.py:19: RuntimeWarning: divide by zero encountered in log
  proba = np.log(lm.get_next_token_prob(prefix, token))


Now let's measure the actual perplexity: we'll split the data into train and test and score model on test data only.

In [10]:
from sklearn.model_selection import train_test_split
train_lines, test_lines = train_test_split(lines, test_size=0.25, random_state=42)

In [12]:
for n in (1, 2, 3):
    lm = NGramLanguageModel(n=n, lines=train_lines)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

100%|██████████| 30750/30750 [00:02<00:00, 11023.82it/s]
C:\Users\lymuthien\AppData\Local\Temp\ipykernel_21560\4221468691.py:19: RuntimeWarning: divide by zero encountered in log
  proba = np.log(lm.get_next_token_prob(prefix, token))


N = 1, Perplexity = 1832.23136


100%|██████████| 30750/30750 [00:04<00:00, 6219.25it/s]


N = 2, Perplexity = 85653987.28544


100%|██████████| 30750/30750 [00:09<00:00, 3134.92it/s]


N = 3, Perplexity = 61999196239895233298432.00000


### LM Smoothing

The problem with our simple language model is that whenever it encounters an n-gram it has never seen before, it assigns it with the probabilitiy of 0. Every time this happens, perplexity explodes.

To battle this issue, there's a technique called __smoothing__. The core idea is to modify counts in a way that prevents probabilities from getting too low. The simplest algorithm here is Additive smoothing (aka [Lapace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing)):

$$ P(w_t | prefix) = { Count(prefix, w_t) + \delta \over \sum_{\hat w} (Count(prefix, \hat w) + \delta) } $$

If counts for a given prefix are low, additive smoothing will adjust probabilities to a more uniform distribution. Not that the summation in the denominator goes over _all words in the vocabulary_.

Here's an example code we've implemented for you:

In [19]:
class LaplaceLanguageModel(NGramLanguageModel): 
    """ this code is an example, no need to change anything """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        self.shift = n - 1
        counts = count_ngrams(lines, self.n)
        self.vocab = set(token for token_counts in counts.values() for token in token_counts)
        self.probs = defaultdict(Counter)

        for prefix in counts:
            token_counts = counts[prefix]
            total_count = sum(token_counts.values()) + delta * len(self.vocab)
            self.probs[prefix] = {token: (token_counts[token] + delta) / total_count
                                          for token in token_counts}
    def get_possible_next_tokens(self, prefix):
        token_probs = super().get_possible_next_tokens(prefix)
        missing_prob_total = 1.0 - sum(token_probs.values())
        missing_prob = missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        return {token: token_probs.get(token, missing_prob) for token in self.vocab}
    
    def get_next_token_prob(self, prefix, next_token):
        token_probs = super().get_possible_next_tokens(prefix)
        if next_token in token_probs:
            return token_probs[next_token]
        else:
            missing_prob_total = 1.0 - sum(token_probs.values())
            missing_prob_total = max(0, missing_prob_total) # prevent rounding errors
            return missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        

**Disclaimer**: the implementation above assumes all words unknown within a given context to be equally likely, *as well as the words outside of vocabulary*. Therefore, its' perplexity will be lower than it should when encountering such words. Therefore, comparing it with a model with fewer unknown words will not be fair. When implementing your own smoothing, you may handle this by adding a virtual `UNK` token of non-zero probability. Technically, this will result in a model where probabilities do not add up to $1$, but it is close enough for a practice excercise.

In [20]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = LaplaceLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

100%|██████████| 100/100 [00:00<00:00, 29920.84it/s]


In [21]:
for n in (1, 2, 3):
    lm = LaplaceLanguageModel(train_lines, n=n, delta=0.1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

100%|██████████| 30750/30750 [00:02<00:00, 12102.68it/s]
C:\Users\lymuthien\AppData\Local\Temp\ipykernel_14216\4221468691.py:19: RuntimeWarning: divide by zero encountered in log
  proba = np.log(lm.get_next_token_prob(prefix, token))


N = 1, Perplexity = 1832.66878


100%|██████████| 30750/30750 [00:04<00:00, 6156.80it/s]


N = 2, Perplexity = 470.48021


100%|██████████| 30750/30750 [00:09<00:00, 3293.11it/s]


N = 3, Perplexity = 3679.44765


In [22]:
# optional: try to sample tokens from such a model

### Kneser-Ney smoothing (2 points)

Additive smoothing is simple, reasonably good but definitely not a State of The Art algorithm.


Your final task in this notebook is to implement [Kneser-Ney](https://en.wikipedia.org/wiki/Kneser%E2%80%93Ney_smoothing) smoothing.

It can be computed recurrently, for n>1:

$$P_{kn}(w_t | prefix_{n-1}) = { \max(0, Count(prefix_{n-1}, w_t) - \delta) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)} + \lambda_{prefix_{n-1}} \cdot P_{kn}(w_t | prefix_{n-2})$$

where
- $prefix_{n-1}$ is a tuple of {n-1} previous tokens
- $lambda_{prefix_{n-1}}$ is a normalization constant chosen so that probabilities add up to 1
- Unigram $P_{kn}(w_t | prefix_{n-2})$ corresponds to Kneser Ney smoothing for {N-1}-gram language model.
- Unigram $P_{kn}(w_t)$ is a special case: how likely it is to see x_t in an unfamiliar context

See lecture slides or wiki for more detailed formulae.

__Your task__ is to
- implement `KneserNeyLanguageModel` class,
- test it on 1-3 gram language models
- find optimal (within reason) smoothing delta for 3-gram language model with Kneser-Ney smoothing

In [13]:
class KneserNeyLanguageModel(NGramLanguageModel): 
    """ A template for Kneser-Ney language model. Default delta may be suboptimal. """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        self.shift = n - 1
        self.delta = delta

        self.probs = defaultdict(Counter)
        self._counts = {k: count_ngrams(lines, k) for k in range(1, n + 1)}
        self.vocab = set()
        for counter in self._counts[1].values():
            self.vocab.update(counter.keys())

        for prefix in self._counts[n]:
            self.probs[prefix] = self._calc_probs(prefix, n)

    def _calc_probs(self, prefix, n):
        if n == 1:
            counter = self._counts[n][()]
            total = counter.total()

            probs = Counter()
            for w in self.vocab:
                probs[w] = counter[w] / total

            return probs

        new_prefix = prefix[-(n - 1):]
        counter = self._counts[n][new_prefix]
        total = counter.total()

        if total == 0:
            return self._calc_probs(prefix[1:], n - 1)

        lambda_ = self.delta * len(counter) / total
        lower = self._calc_probs(prefix[1:], n - 1)
        probs = Counter()

        for token in self.vocab:
            discounted = max(counter[token] - self.delta, 0) / total
            probs[token] = discounted + lambda_ * lower[token]

        return probs

    def get_possible_next_tokens(self, prefix):
        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - self.shift):]
        prefix = [UNK] * (self.shift - len(prefix)) + prefix
        prefix = tuple(prefix)

        if prefix in self.probs:
            return self.probs[prefix]

        if self.n == 1:
            return self._calc_probs((), 1)
        return self._calc_probs(prefix[1:], self.n - 1)

In [11]:
class KneserNeyLanguageModel:
    def __init__(self, lines, n, delta=1):
        self.n = n
        self.shift = n - 1
        self.delta = delta

        self.counts = {i: count_ngrams(lines, i) for i in range(1, n + 1)}
        self.vocab = set(self.counts[1][()].keys())

        self.continuation_counts = Counter()
        if n > 1:
            self._build_continuation_counts()
        else:
            self.continuation_counts = self.counts[1][()]

        self.continuation_total = sum(self.continuation_counts.values())

    def _build_continuation_counts(self):
        for prefix, counter in self.counts[2].items():
            for word in counter:
                self.continuation_counts[word] += 1

    def _prob(self, prefix, word, n):
        if n == 1:
            return self.continuation_counts[word] / self.continuation_total

        counter = self.counts[n].get(prefix, None)
        if counter is None:
            return self._prob(prefix[1:], word, n - 1)

        total = counter.total()
        discounted = max(counter[word] - self.delta, 0) / total
        lambda_ = self.delta * len(counter) / total
        lower_prob = self._prob(prefix[1:], word, n - 1)

        return discounted + lambda_ * lower_prob

    def _get_prefix(self, prefix):
        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - self.shift):]
        prefix = [UNK] * (self.shift - len(prefix)) + prefix

        return tuple(prefix)

    def get_next_token_prob(self, prefix, next_token):
        prefix = self._get_prefix(prefix)

        return self._prob(prefix, next_token, self.n)

    def get_possible_next_tokens(self, prefix):
        return Counter({w: self.get_next_token_prob(prefix, w) for w in self.vocab})

In [14]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = KneserNeyLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

100%|██████████| 100/100 [00:00<00:00, 27255.21it/s]


In [15]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

100%|██████████| 30750/30750 [00:02<00:00, 11251.19it/s]
C:\Users\lymuthien\AppData\Local\Temp\ipykernel_23520\4221468691.py:19: RuntimeWarning: divide by zero encountered in log
  proba = np.log(lm.get_next_token_prob(prefix, token))


N = 1, Perplexity = 1832.23136


100%|██████████| 30750/30750 [00:04<00:00, 6374.83it/s]


N = 2, Perplexity = 385.61602


100%|██████████| 30750/30750 [00:08<00:00, 3463.79it/s]


N = 3, Perplexity = 289.90797
